# Día 5 · Cuaderno 15 — Proyecto integrador: análisis literario con IA **local** (Ollama + Gemma)

**Programar para enseñar — Python e IA generativa para Humanidades Digitales**
Formación docente para EH1023 · Pontificia Universidad Javeriana, Facultad de Ingeniería

*(Cuaderno avanzado / capstone del Día 5)*

---

Este es el cuaderno **más completo** de la semana: reúne **todo lo visto en los cinco días** en
un solo proyecto. La diferencia con los anteriores es que aquí la IA **no vive en otra ventana**
(como TecGPT o Gemini en el Día 4): vamos a **ejecutar un modelo de lenguaje pequeño dentro del
propio cuaderno**, gratis y sin clave de API, usando **Ollama** y el modelo **Gemma 3n (e2b)** de
Google.

**El tema lo eligieron ustedes.** En la encuesta del curso, el interés #1 (16 de 31 respuestas)
fue **el análisis de textos**, y varios pidieron cosas concretas que un LLM resuelve muy bien:

> *"Poder analizar textos literarios/filosóficos con ayuda de Python."*
> *"Análisis de voces de personajes y de estilo de acotaciones en obras dramáticas (teatro)."*
> *"Sentiment analysis en letras de Corridos Tumbados…"*
> *"…identificación de narrativas."*

Por eso el proyecto integrador es un **analizador de textos literarios**: damos un texto y el
modelo nos devuelve **resumen, temas, tono/sentimiento, personajes y lugares** — un análisis
**semántico** que el conteo de palabras de los Días 1–3 **no puede** producir.


## Mapa: dónde reaparece cada día

| Día | Lo que aportó | Dónde se usa aquí |
|---|---|---|
| **Día 1** | Cadenas y análisis de texto | Limpieza del texto y frecuencia de palabras |
| **Día 2** | Funciones, bucles, condicionales, diccionarios | Funciones del pipeline y recorrido del corpus |
| **Día 3** | Archivos, errores, pandas, visualización | Cargar datos, `try/except`, tabla y gráficos |
| **Día 4** | IA generativa: prompts y verificación crítica | Diseño de prompts y control de calidad |
| **Día 5** | APIs, medios y proyecto integrador | El LLM local *es* el servicio; este es el capstone |

> ⚠️ **Antes de empezar:** ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución →
> T4 GPU** y guarda. Sin GPU el modelo funciona, pero va **muy** lento.


## 1. ¿Por qué un modelo **local**? (nota pedagógica)

En el Día 4 usamos IA en otra ventana (TecGPT, Gemini, Copilot). En el Día 5 vimos APIs: pedir
datos a un servicio externo. Hoy juntamos las dos ideas, pero el modelo corre **en la misma
máquina** que el cuaderno. Ventajas que vale la pena discutir con los estudiantes:

- **Privacidad:** el texto **no sale** de tu entorno. Útil con material sensible, inédito o de archivo.
- **Sin costo ni clave:** no hay tarjeta de crédito ni cuota de API.
- **Reproducibilidad:** el mismo modelo da resultados estables; bueno para enseñar y evaluar.
- **Funciona sin conexión** una vez descargado el modelo.

A cambio, un modelo pequeño (como Gemma 3n e2b) es **menos capaz** que los grandes: puede
**equivocarse o inventar** (*alucinar*). Por eso mantenemos **la regla del curso**: la IA propone,
**tú verificas**. Volveremos a esto al final.


## 2. Preparar el entorno: instalar Ollama

**Ollama** es un programa que descarga y ejecuta modelos de lenguaje de forma local. Lo
instalamos en la máquina de Colab con tres comandos. *(Idea tomada del cuaderno
[collama](https://colab.research.google.com/github/5aharsh/collama/blob/main/Ollama_Setup.ipynb).)*

Las celdas que empiezan con `!` ejecutan comandos del sistema, no Python. Tardan 1–2 minutos.


In [ ]:
# Instalar Ollama dentro del entorno de Colab (tarda ~1-2 min)
!sudo apt update -qq
!sudo apt install -y -qq pciutils          # Ollama lo usa para detectar la GPU
!curl -fsSL https://ollama.com/install.sh | sh

### Arrancar Ollama como servicio en segundo plano

Ollama necesita estar **corriendo de fondo** mientras nosotros ejecutamos otras celdas. Como un
cuaderno ejecuta una celda a la vez, lo lanzamos en un **hilo** aparte para que no nos bloquee.
*(No necesitas entender esta celda a fondo; es plomería.)*


In [ ]:
import threading, subprocess, time

def arrancar_ollama():
    subprocess.Popen(["ollama", "serve"])

hilo = threading.Thread(target=arrancar_ollama, daemon=True)
hilo.start()
time.sleep(5)   # darle unos segundos para levantar el servicio
print("Servicio de Ollama en marcha.")

### Descargar el modelo Gemma

Descargamos **Gemma 3n e2b** (~5.6 GB). La primera vez tarda unos minutos; luego queda en caché.

> 🐢 **¿Vas con prisa o sin GPU?** Cambia el nombre del modelo por **`gemma3:1b`** (≈ 0.8 GB):
> descarga y responde mucho más rápido, con algo menos de calidad. Solo cambia la variable
> `MODELO` de la celda siguiente y vuelve a ejecutar.


In [ ]:
# Modelo que usaremos en todo el cuaderno. Cambialo aqui si quieres otro.
MODELO = "gemma3n:e2b"     # alternativa ligera y rapida: "gemma3:1b"

!ollama pull {MODELO}

### Conectar Python con el modelo

Instalamos el cliente de Python de Ollama y creamos una **función ayudante** `preguntar()` que le
manda un texto al modelo y devuelve su respuesta. La usaremos durante todo el cuaderno.


In [ ]:
!pip install -q ollama

In [ ]:
import ollama

def preguntar(prompt, sistema=None, temperatura=0.2):
    """Envia un prompt al modelo local y devuelve el texto de la respuesta.

    - prompt: la instruccion o pregunta (texto).
    - sistema: instruccion de rol opcional (p. ej. "Eres un analista literario").
    - temperatura: 0 = mas predecible, 1 = mas creativo. Para analisis, baja.
    """
    mensajes = []
    if sistema:
        mensajes.append({"role": "system", "content": sistema})
    mensajes.append({"role": "user", "content": prompt})
    respuesta = ollama.chat(
        model=MODELO,
        messages=mensajes,
        options={"temperature": temperatura},
    )
    return respuesta["message"]["content"]

print("Funcion 'preguntar' lista.")

### Prueba de humo

Antes de seguir, confirmemos que el modelo responde. Si ves un párrafo de texto, ¡todo funciona!


In [ ]:
from IPython.display import Markdown, display

prueba = preguntar("Explica en dos frases, para un docente de humanidades, que es un "
                   "modelo de lenguaje. Responde en espanol.")
display(Markdown(prueba))

## 3. Paso 1 · Capturar — cargar el texto (recuerdo del Día 3)

Reutilizamos los datos del curso. Cargamos un **fragmento literario** (el inicio del *Quijote*)
directamente desde el repositorio, igual que hicimos con archivos y URLs en el Día 3. Envolvemos
la descarga en `try/except` (manejo de errores del Día 3): si falla la red, avisamos en lugar de
romper el cuaderno.


In [ ]:
import urllib.request

URL_TEXTO = "https://raw.githubusercontent.com/calderonf/curso-python-humanidades-digitales/main/datos/fragmento_literario.txt"

try:
    with urllib.request.urlopen(URL_TEXTO) as r:
        texto = r.read().decode("utf-8")
    print("Texto cargado:", len(texto), "caracteres\n")
    print(texto[:300], "...")
except Exception as e:
    print("No se pudo descargar. Pega el texto a mano en la variable 'texto'. Detalle:", e)
    texto = "En un lugar de la Mancha, de cuyo nombre no quiero acordarme..."

## 4. Paso 2 · Lo que YA sabes hacer (sin IA todavía)

Con lo de los Días 1–3 podemos **limpiar el texto y contar palabras**. Es útil y rápido, pero
fíjate en **qué nos dice y qué no**.


In [ ]:
import re
from collections import Counter

# Limpieza basica (Dia 1: cadenas): minusculas y solo letras
palabras = re.findall(r"[a-záéíóúüñ]+", texto.lower())

# Palabras vacias que no aportan significado (las quitamos)
vacias = {"de","la","que","el","en","y","a","los","del","se","las","por","un",
          "una","su","mas","no","ha","con","para","es","lo","como","mas","sus",
          "al","le","ya","o","este","si","porque","esta","entre","cuando"}
contenido = [p for p in palabras if p not in vacias and len(p) > 3]

frecuencia = Counter(contenido).most_common(10)
print("Total de palabras:", len(palabras))
print("Palabras de contenido mas frecuentes:")
for palabra, n in frecuencia:
    print(f"  {palabra:15} {n}")

In [ ]:
import matplotlib.pyplot as plt

etiquetas = [p for p, _ in frecuencia]
valores   = [n for _, n in frecuencia]

plt.figure(figsize=(9, 4))
plt.bar(etiquetas, valores)
plt.title("Palabras mas frecuentes en el fragmento")
plt.ylabel("Veces que aparece")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 5. El límite del código tradicional

El gráfico anterior nos dice **qué palabras se repiten**, pero **no** nos dice:

- ¿De qué **trata** el texto?
- ¿Qué **tono** o emoción transmite?
- ¿Quiénes son los **personajes** y los **lugares**?
- ¿A qué **género** literario pertenece?

Eso es **comprensión del lenguaje**, y es justo **el tema complejo** que pidieron en la encuesta.
Aquí es donde un LLM aporta algo que el conteo de palabras no puede. Vamos a ello.


## 6. Paso 3 · Análisis semántico con la IA (recuerdo del Día 4)

Aplicamos la **anatomía de un buen prompt** del Día 4 (contexto, tarea, datos, salida,
restricciones). Le pedimos al modelo que devuelva un **análisis estructurado en JSON** para poder
usarlo después como datos (y no como un párrafo suelto).


In [ ]:
def analizar_texto(fragmento):
    """Pide al modelo un analisis literario estructurado y lo devuelve como texto JSON."""
    sistema = ("Eres un analista literario riguroso. Respondes SIEMPRE en espanol y "
               "SOLO con un objeto JSON valido, sin texto adicional ni explicaciones.")
    prompt = f"""Analiza el siguiente fragmento literario y devuelve un JSON con estas claves:
- "resumen": una frase breve de que trata.
- "temas": lista de 2 a 4 temas principales.
- "tono": una sola palabra para la emocion dominante (p. ej. melancolico, ironico, esperanzador).
- "personajes": lista de personajes mencionados (vacia si no hay).
- "lugares": lista de lugares mencionados (vacia si no hay).
- "genero": tu mejor estimacion (Novela, Poesia, Teatro, Cuento, Ensayo...).

Fragmento:
\"\"\"{fragmento}\"\"\"
"""
    return preguntar(prompt, sistema=sistema, temperatura=0.1)

salida = analizar_texto(texto)
print(salida)

### Convertir la respuesta en datos utilizables

El modelo devuelve texto. A veces lo rodea de comillas o de ```` ```json ````. Escribimos una
función que **extrae el JSON** de forma tolerante (Día 2: funciones; Día 3: manejo de errores) y
lo convierte en un diccionario de Python.


In [ ]:
import json

def extraer_json(texto_modelo):
    """Intenta convertir la respuesta del modelo en un diccionario de Python."""
    try:
        inicio = texto_modelo.index("{")
        fin    = texto_modelo.rindex("}") + 1
        return json.loads(texto_modelo[inicio:fin])
    except (ValueError, json.JSONDecodeError):
        return None   # si no se pudo, devolvemos None y lo manejamos afuera

analisis = extraer_json(salida)

if analisis:
    print("Resumen :", analisis.get("resumen"))
    print("Temas   :", ", ".join(analisis.get("temas", [])))
    print("Tono    :", analisis.get("tono"))
    print("Personajes:", ", ".join(analisis.get("personajes", [])) or "(ninguno)")
    print("Lugares :", ", ".join(analisis.get("lugares", [])) or "(ninguno)")
    print("Genero  :", analisis.get("genero"))
else:
    print("El modelo no devolvio un JSON valido. Vuelve a ejecutar la celda anterior.")
    print("(Con modelos pequenos pasa a veces; bajar la temperatura ayuda.)")

## 7. Paso 4 · Del texto al **corpus**: analizar varios fragmentos

Un solo texto está bien para aprender; el valor real aparece al **procesar muchos**. Armamos un
pequeño **corpus** de fragmentos de distintos géneros y autores (incluye **poesía y teatro**, como
pidieron en la encuesta) y lo recorremos con un **bucle** (Día 2), guardando cada resultado en una
**tabla de pandas** (Día 3).


In [ ]:
corpus = [
    {"obra": "Don Quijote de la Mancha", "autor": "Cervantes",
     "texto": "En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha mucho "
              "tiempo que vivia un hidalgo de los de lanza en astillero, adarga antigua, "
              "rocin flaco y galgo corredor."},
    {"obra": "Veinte poemas de amor", "autor": "Neruda",
     "texto": "Puedo escribir los versos mas tristes esta noche. Escribir, por ejemplo: "
              "La noche esta estrellada, y tiritan, azules, los astros, a lo lejos."},
    {"obra": "Bodas de sangre", "autor": "Lorca",
     "texto": "NOVIO: (entrando) Madre. MADRE: Que. NOVIO: Me voy. MADRE: Adonde. "
              "NOVIO: A la vina. (Va a salir.) MADRE: Espera."},
    {"obra": "Pedro Paramo", "autor": "Rulfo",
     "texto": "Vine a Comala porque me dijeron que aca vivia mi padre, un tal Pedro Paramo. "
              "Mi madre me lo dijo. Y yo le prometi que vendria a verlo en cuanto ella muriera."},
]
print(f"Corpus con {len(corpus)} fragmentos listo.")

In [ ]:
import pandas as pd

filas = []
for item in corpus:
    print(f"Analizando: {item['obra']} ...")
    crudo = analizar_texto(item["texto"])
    datos = extraer_json(crudo)
    if datos is None:
        print("   (no se pudo interpretar; se omite)")
        continue
    filas.append({
        "obra":  item["obra"],
        "autor": item["autor"],
        "genero": datos.get("genero", "?"),
        "tono":   datos.get("tono", "?"),
        "temas":  ", ".join(datos.get("temas", [])),
        "personajes": ", ".join(datos.get("personajes", [])),
    })

df = pd.DataFrame(filas)
df

### Visualizar el corpus (cierre del pipeline)

Convertimos el análisis del modelo en un **gráfico**: cuántos fragmentos hay de cada **tono**.
Así enlazamos todo: **texto → IA → datos → visualización**.


In [ ]:
conteo_tono = df["tono"].str.lower().value_counts()

plt.figure(figsize=(8, 4))
conteo_tono.plot(kind="bar")
plt.title("Tono dominante detectado por la IA en el corpus")
plt.ylabel("Numero de fragmentos")
plt.xlabel("Tono")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 8. Paso 5 · Una síntesis escrita por la IA

Además de clasificar, un LLM puede **redactar**. Le pasamos los temas detectados en todo el corpus
y le pedimos un **párrafo de síntesis** para una ficha de clase. Esto es algo que el conteo de
palabras nunca podría hacer.


In [ ]:
temas_todos = "; ".join(df["temas"].tolist())

prompt_sintesis = f"""Eres un profesor de literatura. A partir de esta lista de temas
detectados en varios fragmentos literarios, escribe un parrafo breve (4-5 frases, en espanol)
que sintetice las preocupaciones comunes de estos textos, en un tono apto para una ficha de clase.

Temas detectados: {temas_todos}"""

sintesis = preguntar(prompt_sintesis, temperatura=0.4)
display(Markdown(sintesis))

## 9. Paso 6 · Verificación crítica (la **regla del curso**)

Un modelo pequeño es rápido y privado, pero **puede equivocarse**. Antes de usar cualquier
resultado en clase, **verifícalo**. Tres comprobaciones rápidas:

1. **¿Inventó personajes o lugares?** Compara la columna `personajes` con el texto real.
2. **¿El género es correcto?** *Bodas de sangre* debería salir como **Teatro**, no Novela.
3. **¿El tono tiene sentido?** Léelo tú y contrástalo.

Si algo falla, las salidas: bajar la **temperatura**, **reformular el prompt**, o usar un **modelo
más grande**. Nunca pegues a tus estudiantes una salida de IA sin leerla.


In [ ]:
# Verificacion semi-automatica: marcar personajes que NO aparecen literalmente en el texto.
for item in corpus:
    crudo = analizar_texto(item["texto"])
    datos = extraer_json(crudo) or {}
    sospechosos = [p for p in datos.get("personajes", [])
                   if p and p.split()[0].lower() not in item["texto"].lower()]
    estado = "revisar -> " + ", ".join(sospechosos) if sospechosos else "ok"
    print(f"{item['obra'][:28]:28} personajes posiblemente inventados: {estado}")

> 🧭 **Nota para docentes.** Esta celda es, en sí misma, una **mini-actividad de pensamiento
> crítico**: en lugar de creerle a la IA, escribimos código que **audita** a la IA. Es una de las
> competencias centrales del EH1023 y conecta con la preocupación que varios marcaron en la
> encuesta: *que los estudiantes dependan demasiado de la IA*. Aquí la IA es el objeto de estudio,
> no el oráculo.


## 10. Cómo llevar esto a tu clase (y adaptarlo)

El mismo pipeline sirve para **muchos** de los proyectos que pidieron en la encuesta; solo cambia
el texto de entrada y las claves del JSON:

- **Letras de canciones / corridos** → análisis de sentimiento y temas (lo pidió alguien explícitamente).
- **Guiones / teatro** → voces de personajes, tono de las acotaciones.
- **Cobertura periodística** → detección de narrativas o sesgos.
- **Textos filosóficos** → extracción de tesis y conceptos clave.

**Calibrar la dificultad** (la idea que atraviesa el curso): para principiantes, dales el pipeline
ya armado y que **solo cambien el texto**. Para avanzados, que **diseñen el JSON** y **escriban la
celda de verificación**.


## 11. Ejercicios graduados

Avanza a tu ritmo. **No es obligatorio llegar al reto.**

- **Nivel 1.** Cambia el texto del primer análisis por un fragmento tuyo (un poema, una noticia, un
  párrafo de tu materia) y vuelve a ejecutar `analizar_texto`.
- **Nivel 2.** Agrega un fragmento más al `corpus` y vuelve a generar la tabla y el gráfico.
- **Reto (opcional).** Añade al prompt una clave nueva, `"sentimiento"`, con valor `positivo`,
  `negativo` o `neutro`, y haz un gráfico de barras de esa columna.


In [ ]:
# Tu solucion del Nivel 1:
mi_texto = "Pega aqui tu propio fragmento..."
# print(analizar_texto(mi_texto))

<details>
<summary>👀 Solución posible del Nivel 1</summary>

```python
mi_texto = ("Puedo escribir los versos mas tristes esta noche. "
            "Pienso que no la tengo. Siento que la he perdido.")
print(analizar_texto(mi_texto))
```
</details>


In [ ]:
# Tu solucion del Nivel 2:
# corpus.append({"obra": "...", "autor": "...", "texto": "..."})
# vuelve a ejecutar las celdas del Paso 4

<details>
<summary>👀 Solución posible del Nivel 2</summary>

```python
corpus.append({
    "obra": "Cien anios de soledad", "autor": "Garcia Marquez",
    "texto": ("Muchos anios despues, frente al peloton de fusilamiento, el coronel "
              "Aureliano Buendia habia de recordar aquella tarde remota en que su "
              "padre lo llevo a conocer el hielo."),
})
# Reejecuta la celda que construye 'df' y la del grafico.
```
</details>


In [ ]:
# Tu solucion del Reto (opcional):
# Pista: copia 'analizar_texto', agrega "sentimiento" a las claves del prompt,
# vuelve a construir el DataFrame y grafica df["sentimiento"].value_counts()

<details>
<summary>👀 Solución posible del Reto</summary>

```python
def analizar_con_sentimiento(fragmento):
    sistema = ("Eres un analista literario. Respondes SOLO con un JSON valido en espanol.")
    prompt = f"""Devuelve un JSON con las claves "tono", "temas" (lista) y
"sentimiento" (uno de: positivo, negativo, neutro).
Fragmento: \"\"\"{fragmento}\"\"\""""
    return extraer_json(preguntar(prompt, sistema=sistema, temperatura=0.1))

filas = []
for item in corpus:
    d = analizar_con_sentimiento(item["texto"]) or {}
    filas.append({"obra": item["obra"], "sentimiento": d.get("sentimiento", "?")})

import pandas as pd
df_sent = pd.DataFrame(filas)
df_sent["sentimiento"].value_counts().plot(kind="bar", title="Sentimiento por fragmento")
plt.tight_layout(); plt.show()
```
</details>


---
## Cierre del curso

En este cuaderno reuniste **toda la semana** en un proyecto real:

- **Cadenas y conteo** de palabras (Día 1) y sus **límites**.
- **Funciones, bucles y diccionarios** (Día 2) para estructurar el pipeline.
- **Archivos, errores, pandas y gráficos** (Día 3) para mover y visualizar datos.
- **Prompts y verificación crítica** (Día 4) para usar la IA con criterio.
- Un **modelo de lenguaje ejecutándose localmente** (Día 5), gratis y privado, como motor del análisis.

Y, sobre todo, practicaste la competencia que da sentido a todo el EH1023: **usar la IA de forma
crítica** — no creerle, sino **verificarla** — y saber **calibrar** la dificultad para tus
estudiantes.

> **Producto para tu portafolio:** este cuaderno, adaptado al tipo de texto de **tu** materia, es
> un material reutilizable listo para llevar al aula.

¡Felicidades por terminar la semana! 🎓
